# cellmap-flow on Colab

Runs `cellmap_flow_server` on this Colab session and exposes it via a
public Cloudflare Tunnel. Builds the cellmap-flow browser dashboard
frontend alongside and exposes it via a second tunnel.

Prints **two URLs** at the end:

- **Bare Neuroglancer link** — vanilla NG with raw + inference layers,
  same as `cellmap_flow huggingface ...` prints on a Janelia workstation.
- **cellmap-flow dashboard URL** — full cellmap-flow chrome with the
  Input/Postprocess pipeline editor and Models picker around an embedded
  NG. Click "Open in NG" once it loads.

Pick whichever fits.

Free Colab gives you a T4 GPU. URLs only alive while this session is.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run all cells.
3. Open one of the printed URLs.

## 1. Install cellmap-flow + cloudflared

Takes ~3–5 min the first time. cloudflared is a tiny static binary;
no signup needed.

In [ ]:
# cellmap-flow with the bioimageio extra (so `bioimage` works too).
%pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs

# Pin bioimageio.spec to a version that still parses v0.4-style RDFs
# (hiding-blowfish + most 2D BMZ models still ship those).
%pip install -q --force-reinstall "bioimageio.core==0.9.6" "bioimageio.spec==0.5.7.4"

# Node.js for building the cellmap-flow dashboard frontend.
!command -v node >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq nodejs npm)
!node --version && npm --version

# cloudflared static binary for the public tunnels.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

# Repo checkout for the browser/ frontend.
import os, subprocess
REPO_DIR = "/content/cellmap-flow-repo"
if not os.path.isdir(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth=1", "--branch=browser-inference",
        "https://github.com/janelia-cellmap/cellmap-flow.git", REPO_DIR,
    ])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--depth=1", "origin", "browser-inference"])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", "origin/browser-inference"])
print("repo at:", REPO_DIR)


## 2. Configure model + dataset

Pick one of two model types:
- `huggingface` — a cellmap HF model (e.g. `cellmap/fly_organelles_run07_432000`).
- `bioimage` — a BioImage Model Zoo model (e.g. `hiding-blowfish`).

Set `MODEL_TYPE` to switch.

> **T4 sizing**: 178³-input HF models (`fly_organelles_run07_*`) fit T4
> cleanly (~5–6 GB peak). 288³ models (`jrc_mus-livers_*`) are
> borderline. 378³ models (`mito-aff-unet-*`) won't fit. BMZ 2D models
> like `hiding-blowfish` are tiny (~200 MB peak).

In [ ]:
MODEL_TYPE = "huggingface"   # or "bioimage"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"  # 178^3 inference; fits T4
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)
# Other T4-friendly options:
#   HF_REPO = "cellmap/fly_organelles_run07_700000"
#   HF_REPO = "cellmap/fly_organelles_run08_438000"
#   HF_REPO = "cellmap/jrc_mus-livers_16nm_to_8nm_mito"  # BORDERLINE, OOM-prone

# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"     # nm per voxel
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

PORT = 8765

if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")


## 3. Start the cellmap-flow inference server

Runs in the background so cloudflared can come up alongside.

In [ ]:
import os, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO,
        "--name", HF_NAME,
        "-d", HF_DATASET,
        "--port", str(PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL,
        "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET,
        "--port", str(PORT),
    ]

print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{PORT}" in line:
        print("\n[server] ready.")
        break


## 4. Public cloudflared tunnel

In [ ]:
import subprocess, re, time

tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break

print("\n" + "=" * 70)
print(f"BACKEND URL: {PUBLIC_URL}")
print("=" * 70)


## 5. Build the cellmap-flow dashboard frontend

Compiles `browser/dist/` from the repo checkout. Takes ~1–2 minutes on
first run; cached on re-runs.

In [ ]:
import subprocess, os
os.chdir(f"{REPO_DIR}/browser")
subprocess.check_call(["npm", "install", "--no-audit", "--no-fund", "--no-progress"])
subprocess.check_call(["npm", "run", "build"])
os.chdir("/content")
print("frontend built at:", f"{REPO_DIR}/browser/dist")


## 6. Serve the frontend + cloudflared tunnel for it

In [ ]:
import subprocess, time, socket, re

FRONTEND_PORT = 4173

# Kill any stale http.server on this port (re-runs).
for p in subprocess.run(["pgrep", "-f", "http.server.*--directory.*browser/dist"],
                        capture_output=True, text=True).stdout.split():
    subprocess.run(["kill", p])
time.sleep(0.5)

frontend = subprocess.Popen(
    ["python3", "-m", "http.server", str(FRONTEND_PORT),
     "--directory", f"{REPO_DIR}/browser/dist"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for _ in range(40):
    try:
        with socket.create_connection(("127.0.0.1", FRONTEND_PORT), timeout=0.5):
            print(f"frontend listening on :{FRONTEND_PORT}")
            break
    except OSError:
        time.sleep(0.25)
else:
    raise RuntimeError("frontend http.server didn't bind in time")

tunnel_fe = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{FRONTEND_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
FRONTEND_URL = None
for _ in range(120):
    line = tunnel_fe.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        FRONTEND_URL = m.group(0)
        break
print(f"\nFRONTEND URL: {FRONTEND_URL}")


## 7. Build the demo URLs

Two options — pick whichever fits.


In [ ]:
import json, urllib.parse, re

# Bare-NG URL (path 1: simple, no chrome).
raw_url = re.sub(r"/s\d+/?$", "", DATASET)
voxel_nm = 8
state = {
    "dimensions": {
        "z": [voxel_nm * 1e-9, "m"],
        "y": [voxel_nm * 1e-9, "m"],
        "x": [voxel_nm * 1e-9, "m"],
    },
    "layers": [
        {"type": "image", "source": f"zarr://{raw_url}", "name": "raw"},
        {
            "type": "image",
            "source": f"zarr://{PUBLIC_URL}/{MODEL_NAME}/",
            "name": MODEL_NAME,
        },
    ],
    "selectedLayer": {"visible": True, "layer": MODEL_NAME},
    "layout": "4panel",
}
encoded = urllib.parse.quote(json.dumps(state, separators=(",", ":")))
ng_url = f"https://neuroglancer-demo.appspot.com/#!{encoded}"

# Dashboard URL (path 2: cellmap-flow chrome around the embedded NG).
qp = {
    "backend": PUBLIC_URL,
    "dataset": MODEL_NAME,
    "raw": raw_url,
}
if MODEL_TYPE == "huggingface":
    qp["hf"] = HF_REPO
qs = urllib.parse.urlencode(qp, safe=":/")
dashboard_url = f"{FRONTEND_URL}/dashboard.html?{qs}"

print()
print("=" * 70)
print("OPTION A — bare Neuroglancer (vanilla NG, no cellmap-flow chrome):")
print(ng_url)
print()
print("OPTION B — cellmap-flow dashboard (full UI w/ Input/Postprocess editor):")
print("  open this, then click 'Open in NG' in the center panel.")
print(dashboard_url)
print("=" * 70)
print()
print("Direct cellmap-flow inference server (for other tools):")
print(f"  {PUBLIC_URL}")


## 8. Keep-alive

Run last so Colab doesn't idle-disconnect. Stop with ▢ to tear down.


In [ ]:
import time, select

procs = [(server, "server"), (tunnel, "tunnel-backend"),
         (frontend, "frontend"), (tunnel_fe, "tunnel-frontend")]

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r: return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        died = False
        for p, label in procs:
            drain(p, label)
            if p.poll() is not None:
                drain(p, label)
                print(f"\n[{label}] exited rc={p.returncode}.")
                died = True
        if died: break
        time.sleep(2)
finally:
    for p, _ in procs:
        try: p.terminate()
        except Exception: pass
